In [1]:
%pip install -q crewai langgraph langchain-google-genai langchain-google-vertexai langchain-community mlflow pandas "ragas==0.3.9"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.7/366.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
from getpass import getpass

os.environ.pop("GOOGLE_API_KEY", None)
os.environ.pop("GEMINI_API_KEY", None)

gemini_key = getpass("Enter your Gemini API key: ")

os.environ["GOOGLE_API_KEY"] = gemini_key
os.environ["GEMINI_API_KEY"] = gemini_key

print("Gemini API key configured.")

Enter your Gemini API key: ··········
Gemini API key configured.


In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.2
)

test = llm.invoke("Reply with exactly: Gemini test successful")
print(test.content)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3914: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Gemini test successful', 'extras': {'signature': 'EmAKXgFpFH0TVtEpsD+QeAYjBwZP+/2Rs7mZmM0mhmmbhkfdGRpFMKlEjZsXHLCeJGA+mQhxOUUbpNDjKIeQazAm5iwkqkUCm4dFjxLUB615hx5KviAnjdWZ9l+dJ3kzHNY='}}]


In [4]:
import pandas as pd
import mlflow

from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

print("LangGraph, pandas, and MLflow imported successfully.")

LangGraph, pandas, and MLflow imported successfully.


In [5]:
import sys
import langchain_google_vertexai.chat_models as vertex_models

sys.modules[
    "langchain_community.chat_models.vertexai"
] = vertex_models

print("Ragas Vertex AI compatibility patch applied.")

Ragas Vertex AI compatibility patch applied.


In [6]:
import ragas

from ragas import EvaluationDataset, evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

print("Ragas version:", ragas.__version__)
print("Ragas evaluation imports successful.")

Ragas version: 0.3.9
Ragas evaluation imports successful.


In [7]:
class SupportState(TypedDict):
    customer_id: str
    messages: List[dict]
    issue: str
    response: str
    resolved: bool

print("Customer support state defined successfully.")

Customer support state defined successfully.


In [8]:
def support_agent(state: SupportState):
    history = state["messages"]

    conversation = "\n".join(
        f'{item["role"]}: {item["content"]}'
        for item in history
    )

    prompt = f"""
You are a polite online-store customer support agent.

Use the full conversation history for follow-up questions.
Never invent order status, delivery updates, or refund status.
If information is missing, ask the customer for it.
Do not claim that an issue is resolved unless confirmed.

Customer ID: {state["customer_id"]}
Issue: {state["issue"]}

Conversation history:
{conversation}

Write the next customer support response.
"""

    result = llm.invoke(prompt)

    answer = result.content
    if isinstance(answer, list):
        answer = " ".join(
            item.get("text", "") if isinstance(item, dict)
            else str(item)
            for item in answer
        )
    answer = str(answer)

    return {
        "response": answer,
        "messages": history + [
            {"role": "assistant", "content": answer}
        ]
    }

print("Customer support agent defined successfully.")

Customer support agent defined successfully.


In [9]:
graph_builder = StateGraph(SupportState)

graph_builder.add_node("support_agent", support_agent)
graph_builder.add_edge(START, "support_agent")
graph_builder.add_edge("support_agent", END)

support_graph = graph_builder.compile()

print("LangGraph workflow compiled successfully.")

LangGraph workflow compiled successfully.


In [14]:
initial_state = {
    "customer_id": "CUST001",
    "messages": [
        {
            "role": "user",
            "content": "My order ORD123 has not arrived. Can you help?"
        }
    ],
    "issue": "Delayed delivery",
    "response": "",
    "resolved": False
}

result1 = support_graph.invoke(initial_state)

print("Customer:", initial_state["customer_id"])
print("Agent:", result1["response"])
print("Messages stored:", len(result1["messages"]))

assert result1["response"].strip()
assert len(result1["messages"]) == 2

print("PASS: First support interaction completed.")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3914: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Customer: CUST001
Agent: Hello! I would be happy to help you look into your order ORD123. 

To help me check the current status of your delivery, could you please confirm your shipping address and the email address associated with your account?
Messages stored: 2
PASS: First support interaction completed.


In [15]:
followup_state = {
    **initial_state,
    "messages": result1["messages"] + [
        {
            "role": "user",
            "content": "What was my order number?"
        }
    ]
}

result2 = support_graph.invoke(followup_state)

print("Follow-up:", followup_state["messages"][-1]["content"])
print("Agent:", result2["response"])
print("Messages stored:", len(result2["messages"]))

assert len(result2["messages"]) == 4
assert "ORD123" in result2["response"]

print("PASS: Previous conversation was preserved.")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3914: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Follow-up: What was my order number?
Agent: Hello! 

Your order number is ORD123. 

To help me look into the delivery status for you, could you please confirm the shipping address and the email address associated with your account?
Messages stored: 4
PASS: Previous conversation was preserved.


In [16]:
from crewai import Agent, Task, Crew, Process, LLM

crew_llm = LLM(
    model="gemini/gemini-3.5-flash-lite",
    api_key=gemini_key,
    temperature=0.2
)

reviewer = Agent(
    role="Customer Support Quality Reviewer",
    goal="Review support responses for accuracy, empathy, and clarity",
    backstory=(
        "You review online-store support responses. "
        "You prevent invented delivery or refund claims."
    ),
    llm=crew_llm,
    allow_delegation=False,
    verbose=False
)

print("CrewAI reviewer configured successfully.")

CrewAI reviewer configured successfully.


In [20]:
review_task = Task(
    description=f"""
Review this customer support response.

Customer issue: {initial_state["issue"]}
Customer message: {initial_state["messages"][0]["content"]}
Support response: {result1["response"]}

Check:
1. Is the response polite and empathetic?
2. Does it avoid inventing delivery status?
3. Does it give a useful next step?

Return a short review and one improvement if needed.
""",
    expected_output="A short quality review with an improvement if necessary.",
    agent=reviewer
)

review_crew = Crew(
    agents=[reviewer],
    tasks=[review_task],
    process=Process.sequential,
    verbose=False
)

crew_review = await review_crew.kickoff_async()

print("CrewAI quality review:")
print(crew_review)

CrewAI quality review:
**Quality Review:**

1. **Polite and empathetic:** Yes, the response opens with a friendly, welcoming, and helpful tone. 
2. **Avoids inventing delivery status:** Yes, the agent successfully refrains from guessing or making up any false claims about where the package is or when it will arrive.
3. **Gives a useful next step:** Yes, asking for the shipping address and associated email is a logical and necessary next step to securely locate the customer's account and order details.

**Improvement:**
To make the response even more empathetic and reassuring, acknowledge the delay upfront before asking for verification. 

*Suggested improvement:* 
"Hello! I am so sorry to hear that your order ORD123 has not arrived yet. I would be happy to help you look into this. To help me check the current status of your delivery, could you please confirm your shipping address and the email address associated with your account?"


In [21]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("W10D5_Customer_Support")

print("MLflow experiment configured.")
print("Tracking URI:", mlflow.get_tracking_uri())

2026/10/13 10:38:36 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/13 10:38:36 INFO mlflow.store.db.utils: Updating database tables
2026/10/13 10:38:41 INFO mlflow.tracking.fluent: Experiment with name 'W10D5_Customer_Support' does not exist. Creating a new experiment.


MLflow experiment configured.
Tracking URI: sqlite:///mlflow.db


In [22]:
with mlflow.start_run(run_name="stateful_support_agent"):
    mlflow.log_param("customer_id", initial_state["customer_id"])
    mlflow.log_param("issue", initial_state["issue"])
    mlflow.log_param("framework", "LangGraph + CrewAI + Gemini")

    mlflow.log_metric(
        "messages_after_first_turn",
        len(result1["messages"])
    )
    mlflow.log_metric(
        "messages_after_followup",
        len(result2["messages"])
    )

    mlflow.set_tag("project", "W10D5 Stateful Customer Support Agent")
    mlflow.set_tag("test", "conversation_memory")

print("MLflow run logged successfully.")

MLflow run logged successfully.


In [23]:
def clean_answer(value):
    if isinstance(value, list):
        return " ".join(
            item.get("text", "") if isinstance(item, dict)
            else str(item)
            for item in value
        )
    return str(value)

evaluation_data = pd.DataFrame([
    {
        "question": initial_state["messages"][0]["content"],
        "answer": clean_answer(result1["response"]),
        "reference": (
            "The agent should acknowledge the delayed order "
            "and explain that delivery status must be checked."
        ),
        "contexts": [
            "For delayed orders, acknowledge the concern and "
            "explain that delivery status must be checked."
        ]
    },
    {
        "question": "What was my order number?",
        "answer": clean_answer(result2["response"]),
        "reference": "The order number mentioned earlier was ORD123.",
        "contexts": [
            "The customer previously stated that the order number is ORD123."
        ]
    }
])

print("Evaluation rows:", len(evaluation_data))
print("Missing values:", evaluation_data.isnull().sum().sum())

display(evaluation_data[["question", "answer", "reference"]])

assert len(evaluation_data) == 2
assert evaluation_data.isnull().sum().sum() == 0

Evaluation rows: 2
Missing values: 0


,question,answer,reference
0,My order ORD123 has not arrived. Can you help?,Hello! I would be happy to help you look into ...,The agent should acknowledge the delayed order...
1,What was my order number?,Hello! \n\nYour order number is ORD123. \n\nTo...,The order number mentioned earlier was ORD123.


In [24]:
ragas_dataset = EvaluationDataset.from_list([
    {
        "user_input": row["question"],
        "response": row["answer"],
        "retrieved_contexts": row["contexts"],
        "reference": row["reference"]
    }
    for _, row in evaluation_data.iterrows()
])

print("Ragas dataset created.")
print("Evaluation samples:", len(ragas_dataset))

Ragas dataset created.
Evaluation samples: 2


In [25]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

run_config = RunConfig(
    timeout=300,
    max_retries=1,
    max_wait=5,
    max_workers=1
)

evaluator_llm = LangchainLLMWrapper(
    llm,
    run_config=run_config
)

gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

evaluator_embeddings = LangchainEmbeddingsWrapper(
    gemini_embeddings
)

print("Ragas Gemini evaluator configured.")
print("Timeout:", run_config.timeout, "seconds")
print("Concurrent workers:", run_config.max_workers)

/tmp/ipykernel_346/2844837987.py:10: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(


Ragas Gemini evaluator configured.
Timeout: 300 seconds
Concurrent workers: 1


/tmp/ipykernel_346/2844837987.py:19: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(


In [26]:
test_dataset = EvaluationDataset.from_list([
    {
        "user_input": ragas_dataset.samples[0].user_input,
        "response": ragas_dataset.samples[0].response,
        "retrieved_contexts": ragas_dataset.samples[0].retrieved_contexts,
        "reference": ragas_dataset.samples[0].reference
    }
])

test_results = evaluate(
    dataset=test_dataset,
    metrics=[Faithfulness()],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=run_config,
    raise_exceptions=True
)

test_df = test_results.to_pandas()
display(test_df)

assert test_df["faithfulness"].notna().all(), (
    "Faithfulness evaluation returned a missing score."
)

print("PASS: Single-sample Ragas test succeeded.")

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3914: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3914: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


,user_input,retrieved_contexts,response,reference,faithfulness
0,My order ORD123 has not arrived. Can you help?,"[For delayed orders, acknowledge the concern a...",Hello! I would be happy to help you look into ...,The agent should acknowledge the delayed order...,0.0


PASS: Single-sample Ragas test succeeded.


In [30]:
from ragas import evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy

# Use one generated candidate for Gemini compatibility
metrics = [
    Faithfulness(),
    AnswerRelevancy(strictness=1)
]

ragas_results = evaluate(
    dataset=ragas_dataset,
    metrics=metrics,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=run_config,
    raise_exceptions=True
)

ragas_df = ragas_results.to_pandas()

display(
    ragas_df[["user_input", "faithfulness", "answer_relevancy"]]
)

assert ragas_df[
    ["faithfulness", "answer_relevancy"]
].notna().all().all()

print("PASS: Ragas evaluation completed successfully.")

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3914: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3914: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3914: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3914: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

,user_input,faithfulness,answer_relevancy
0,My order ORD123 has not arrived. Can you help?,0.0,0.813508
1,What was my order number?,1.0,0.740931


PASS: Ragas evaluation completed successfully.


In [31]:
ragas_df.to_csv("w10d5_ragas_results.csv", index=False)

print("Saved: w10d5_ragas_results.csv")
display(ragas_df)

Saved: w10d5_ragas_results.csv


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy
0,My order ORD123 has not arrived. Can you help?,"[For delayed orders, acknowledge the concern a...",Hello! I would be happy to help you look into ...,The agent should acknowledge the delayed order...,0.0,0.813508
1,What was my order number?,[The customer previously stated that the order...,Hello! \n\nYour order number is ORD123. \n\nTo...,The order number mentioned earlier was ORD123.,1.0,0.740931


In [32]:
with mlflow.start_run(run_name="ragas_evaluation"):
    mlflow.log_param("evaluation_samples", len(ragas_df))
    mlflow.log_param("metrics", "faithfulness, answer_relevancy")

    mlflow.log_metric(
        "mean_faithfulness",
        float(ragas_df["faithfulness"].mean())
    )

    mlflow.log_metric(
        "mean_answer_relevancy",
        float(ragas_df["answer_relevancy"].mean())
    )

    mlflow.log_artifact("w10d5_ragas_results.csv")
    mlflow.set_tag("evaluation_status", "success")

print("Ragas scores logged to MLflow.")

ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-22' coro=<as_completed.<locals>.sema_coro() done, defined at /usr/local/lib/python3.13/dist-packages/ragas/async_utils.py:75> exception=AssertionError('llm must be set to compute score')>
Traceback (most recent call last):
  File "/usr/lib/python3.13/asyncio/tasks.py", line 313, in __step_run_and_handle_result
    result = coro.send(None)
  File "/usr/local/lib/python3.13/dist-packages/ragas/async_utils.py", line 77, in sema_coro
    return await coro
           ^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/ragas/executor.py", line 73, in wrapped_callable_async
    raise e
  File "/usr/local/lib/python3.13/dist-packages/ragas/executor.py", line 69, in wrapped_callable_async
    result = await callable(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/ragas/metrics/base.py", line 488, in single_turn_ascore
    raise e
  File "/usr/lo

Ragas scores logged to MLflow.


In [33]:
assert result1["response"].strip()
assert result2["response"].strip()
assert "ORD123" in result2["response"]
assert len(result1["messages"]) == 2
assert len(result2["messages"]) == 4

assert len(ragas_df) == 2
assert ragas_df[
    ["faithfulness", "answer_relevancy"]
].notna().all().all()

print("PASS: Support responses generated.")
print("PASS: Conversation memory verified.")
print("PASS: Ragas scores validated.")
print("PASS: MLflow tracking configured.")
print("W10D5 final validation completed.")

PASS: Support responses generated.
PASS: Conversation memory verified.
PASS: Ragas scores validated.
PASS: MLflow tracking configured.
W10D5 final validation completed.


In [34]:
from pathlib import Path

readme = """# W10D5: Stateful Customer Support Agent

## Objective
Build and test a stateful customer support agent.

## Technologies
- CrewAI: response quality review
- LangGraph: stateful conversation workflow
- Gemini API: language model
- MLflow: experiment tracking
- Ragas: faithfulness and answer relevancy
- pandas: evaluation dataset and results

## Features
- Customer support responses
- Conversation history preservation
- Follow-up question testing
- CrewAI quality review
- Ragas evaluation
- MLflow metric tracking

## Run
1. Open the notebook in Google Colab.
2. Install dependencies.
3. Configure the Gemini API key securely.
4. Run the cells in order.

Never commit API keys or private credentials.
"""

requirements = """crewai
langgraph
langchain-google-genai
langchain-google-vertexai
langchain-community
mlflow
pandas
ragas==0.3.9
"""

Path("README_W10D5.md").write_text(readme, encoding="utf-8")
Path("requirements_W10D5.txt").write_text(requirements, encoding="utf-8")

print("Created README_W10D5.md")
print("Created requirements_W10D5.txt")

Created README_W10D5.md
Created requirements_W10D5.txt


In [35]:
from pathlib import Path

for filename in [
    "README_W10D5.md",
    "requirements_W10D5.txt",
    "w10d5_ragas_results.csv"
]:
    print(
        filename,
        "—",
        "FOUND" if Path(filename).exists() else "NOT CREATED"
    )

README_W10D5.md — FOUND
requirements_W10D5.txt — FOUND
w10d5_ragas_results.csv — FOUND


In [36]:
from google.colab import files
files.download("requirements_W10D5.txt")

from google.colab import files
files.download("w10d5_ragas_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>